In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
spark = SparkSession.builder.appName("shuffle-practice").master("local[*]").getOrCreate()
print(spark.version)

4.2.0


In [3]:
bookings = spark.createDataFrame(
    [(1, "ORD-LHR", "Economy", 640.0),
     (2, "ORD-LHR", "Business", 2900.0),
     (3, "ORD-FRA", "Economy", 590.0),
     (4, "ORD-FRA", "Economy", 610.0),
     (5, "ORD-CDG", "Business", 3100.0)],
    ["booking_id", "route", "cabin", "fare"])

bookings.show()

+----------+-------+--------+------+
|booking_id|  route|   cabin|  fare|
+----------+-------+--------+------+
|         1|ORD-LHR| Economy| 640.0|
|         2|ORD-LHR|Business|2900.0|
|         3|ORD-FRA| Economy| 590.0|
|         4|ORD-FRA| Economy| 610.0|
|         5|ORD-CDG|Business|3100.0|
+----------+-------+--------+------+



In [4]:
aggdf = bookings.groupBy("route").agg(sum("fare").alias("revenue"), count("*").alias("bookings"))

In [5]:
aggdf.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[route#1], functions=[sum(fare#3), count(1)])
   +- Exchange hashpartitioning(route#1, 200), ENSURE_REQUIREMENTS, [plan_id=27]
      +- HashAggregate(keys=[route#1], functions=[partial_sum(fare#3), partial_count(1)])
         +- Project [route#1, fare#3]
            +- Scan ExistingRDD[booking_id#0L,route#1,cabin#2,fare#3]




In [7]:
aggdf.show()

+-------+-------+--------+
|  route|revenue|bookings|
+-------+-------+--------+
|ORD-LHR| 3540.0|       2|
|ORD-FRA| 1200.0|       2|
|ORD-CDG| 3100.0|       1|
+-------+-------+--------+



In [10]:
big = (spark.range(0, 5_000_000)
       .withColumn("bucket", col("id") % 100)
       .groupBy("bucket")
       .count())
big.collect()

[Row(bucket=26, count=50000),
 Row(bucket=29, count=50000),
 Row(bucket=65, count=50000),
 Row(bucket=19, count=50000),
 Row(bucket=54, count=50000),
 Row(bucket=0, count=50000),
 Row(bucket=22, count=50000),
 Row(bucket=7, count=50000),
 Row(bucket=77, count=50000),
 Row(bucket=34, count=50000),
 Row(bucket=50, count=50000),
 Row(bucket=94, count=50000),
 Row(bucket=57, count=50000),
 Row(bucket=32, count=50000),
 Row(bucket=43, count=50000),
 Row(bucket=84, count=50000),
 Row(bucket=31, count=50000),
 Row(bucket=39, count=50000),
 Row(bucket=98, count=50000),
 Row(bucket=25, count=50000),
 Row(bucket=95, count=50000),
 Row(bucket=71, count=50000),
 Row(bucket=6, count=50000),
 Row(bucket=68, count=50000),
 Row(bucket=72, count=50000),
 Row(bucket=87, count=50000),
 Row(bucket=58, count=50000),
 Row(bucket=9, count=50000),
 Row(bucket=27, count=50000),
 Row(bucket=63, count=50000),
 Row(bucket=51, count=50000),
 Row(bucket=56, count=50000),
 Row(bucket=52, count=50000),
 Row(bucket=17

In [11]:
spark.sparkContext.defaultParallelism


8

In [12]:
import os
print(os.listdir("/home/jovyan/work"))

['shuffle-practice.ipynb', '.ipynb_checkpoints']


Why groupBy creates a stage

A stage is a set of tasks that can run without needing data from other tasks. Steps like range, withColumn, and filter work on each row alone, so each task handles its own partition with no talking to others.
groupBy needs all rows with the same key in the same place. Those rows start out spread across partitions, so data has to move between tasks. Spark ends the stage there and starts a new one.
So a stage boundary is a shuffle boundary.

Shuffle write and shuffle read

Shuffle write happens at the end of stage 1. Each task counts its own rows first (a partial count, which keeps the shuffled data small), sorts the results into buckets by key, and writes them to local disk.
Shuffle read happens at the start of stage 2. Each task fetches its bucket’s pieces from every stage 1 task.
In the UI, stage 1 shows Shuffle Write and stage 2 shows Shuffle Read, and the two sizes should be roughly equal.

What the buckets are

A bucket is a shuffle partition: a destination that a set of keys is sent to.
The bucket for a row is decided by hashing the key and taking the remainder after dividing by the number of buckets. The same key always lands in the same bucket, so one task can add up all its pieces.
The number of buckets is spark.sql.shuffle.partitions (default 200), and each bucket gets one stage 2 task.
AQE can merge small buckets, which is why your first run showed 1 task in stage 2.

Stage 1 task count depends on the number of input partitions. During shuffle, each Stage 1 task writes data into all shuffle buckets. The number of shuffle buckets equals the number of shuffle partitions, and each shuffle partition becomes one task in Stage 2. Thus, Stage 1 tasks are based on input partitions, while Stage 2 tasks are based on shuffle partitions.

In [4]:
def run_big():
    return (spark.range(0, 5_000_000)
            .withColumn("key", col("id") % 100)
            .groupBy("key").count().collect())

In [5]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "4")
run_big()

[Row(key=2, count=50000),
 Row(key=4, count=50000),
 Row(key=5, count=50000),
 Row(key=8, count=50000),
 Row(key=12, count=50000),
 Row(key=15, count=50000),
 Row(key=19, count=50000),
 Row(key=23, count=50000),
 Row(key=25, count=50000),
 Row(key=26, count=50000),
 Row(key=28, count=50000),
 Row(key=29, count=50000),
 Row(key=30, count=50000),
 Row(key=31, count=50000),
 Row(key=33, count=50000),
 Row(key=34, count=50000),
 Row(key=42, count=50000),
 Row(key=48, count=50000),
 Row(key=54, count=50000),
 Row(key=61, count=50000),
 Row(key=62, count=50000),
 Row(key=67, count=50000),
 Row(key=73, count=50000),
 Row(key=81, count=50000),
 Row(key=88, count=50000),
 Row(key=91, count=50000),
 Row(key=92, count=50000),
 Row(key=93, count=50000),
 Row(key=0, count=50000),
 Row(key=1, count=50000),
 Row(key=6, count=50000),
 Row(key=9, count=50000),
 Row(key=10, count=50000),
 Row(key=11, count=50000),
 Row(key=22, count=50000),
 Row(key=32, count=50000),
 Row(key=36, count=50000),
 Row(key=

In [6]:
spark.conf.set("spark.sql.shuffle.partitions", "200")
run_big()

[Row(key=26, count=50000),
 Row(key=29, count=50000),
 Row(key=65, count=50000),
 Row(key=19, count=50000),
 Row(key=54, count=50000),
 Row(key=0, count=50000),
 Row(key=22, count=50000),
 Row(key=7, count=50000),
 Row(key=77, count=50000),
 Row(key=34, count=50000),
 Row(key=50, count=50000),
 Row(key=94, count=50000),
 Row(key=57, count=50000),
 Row(key=32, count=50000),
 Row(key=43, count=50000),
 Row(key=84, count=50000),
 Row(key=31, count=50000),
 Row(key=39, count=50000),
 Row(key=98, count=50000),
 Row(key=25, count=50000),
 Row(key=95, count=50000),
 Row(key=71, count=50000),
 Row(key=6, count=50000),
 Row(key=68, count=50000),
 Row(key=72, count=50000),
 Row(key=87, count=50000),
 Row(key=58, count=50000),
 Row(key=9, count=50000),
 Row(key=27, count=50000),
 Row(key=63, count=50000),
 Row(key=51, count=50000),
 Row(key=56, count=50000),
 Row(key=52, count=50000),
 Row(key=17, count=50000),
 Row(key=79, count=50000),
 Row(key=41, count=50000),
 Row(key=28, count=50000),
 Row(

In [7]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")

In [8]:
run_big()

[Row(key=26, count=50000),
 Row(key=29, count=50000),
 Row(key=65, count=50000),
 Row(key=19, count=50000),
 Row(key=54, count=50000),
 Row(key=0, count=50000),
 Row(key=22, count=50000),
 Row(key=7, count=50000),
 Row(key=77, count=50000),
 Row(key=34, count=50000),
 Row(key=50, count=50000),
 Row(key=94, count=50000),
 Row(key=57, count=50000),
 Row(key=32, count=50000),
 Row(key=43, count=50000),
 Row(key=84, count=50000),
 Row(key=31, count=50000),
 Row(key=39, count=50000),
 Row(key=98, count=50000),
 Row(key=25, count=50000),
 Row(key=95, count=50000),
 Row(key=71, count=50000),
 Row(key=6, count=50000),
 Row(key=68, count=50000),
 Row(key=72, count=50000),
 Row(key=87, count=50000),
 Row(key=58, count=50000),
 Row(key=9, count=50000),
 Row(key=27, count=50000),
 Row(key=63, count=50000),
 Row(key=51, count=50000),
 Row(key=56, count=50000),
 Row(key=52, count=50000),
 Row(key=17, count=50000),
 Row(key=79, count=50000),
 Row(key=41, count=50000),
 Row(key=28, count=50000),
 Row(